# Tome 10, Chapter 11: Mod-$\varphi$ convergence (Python companion)

Exercises first, solutions at the end. We count the cycles of random permutations and the prime factors of integers, and watch the Poisson law fail in the tails exactly as the residues $e^{-\gamma(x-1)}/\Gamma(x)$ and $\Psi_{\mathrm{unif}}(x)$ predict. We check the asymptotics of the Stirling numbers. The computer science application is the **search for irreducible polynomials over $\mathbb F_2$**, the first step of every implementation of a finite field $\mathbb F_{2^n}$ (used in AES, CRCs and error-correcting codes): the factorisation pattern of a random polynomial behaves like the cycle pattern of a random permutation, so about one try in $n$ succeeds. Requires `numpy` and `scipy`.

In [ ]:
import math, random
import numpy as np
from scipy.special import gamma as Gamma, gammaln
rng = np.random.default_rng(110)
random.seed(110)
EG = 0.5772156649015329   # Euler's constant
def poisson(lam, k):
    return math.exp(-lam + k*math.log(lam) - gammaln(k+1))

## Exercises

**P11.1 (★) Cycles of a random permutation.** Draw $20\,000$ uniform permutations of size $n=1000$ (`rng.permutation`) and count their cycles $K_n$. Compute also the exact law of $K_n$ (Feller's coupling: a sum of independent Bernoulli$(1/j)$). Compare, for each $k$, the empirical frequency, the exact probability, the Poisson probability $\pi_{H_n}(k)$, and the corrected value $\pi_{H_n}(k)\,e^{-\gamma(x-1)}/\Gamma(x)$ with $x=k/H_n$ (Corollary 11.9). Where is the Poisson law wrong, and by how much?

**P11.2 (★★) Stirling numbers of the first kind.** Compute $|s(n,k)|$ exactly for $n=2000$ from $z(z+1)\cdots(z+n-1)$ (Python integers), and compare with $\frac{(n-1)!\,(\log n)^{k-1}}{(k-1)!\,\Gamma(1+k/\log n)}$ for $k=2,\dots,20$. How fast does the ratio approach $1$? Replace $\log n$ by $H_n$ in the Poisson part, as in the corollary: is it better?

**P11.3 (★★) Prime factors.** With a sieve, compute $\omega(n)$ for all $n\le10^7$. Compare the proportions of $n$ with $\omega(n)=k$ with the Poisson probabilities $\pi_\lambda(k)$, $\lambda=\log\log10^7$, and with the Selberg–Delange prediction $\pi_\lambda(k)\Psi_{\mathrm{unif}}(k/\lambda)$, where
$$\Psi_{\mathrm{unif}}(z)=\frac1{\Gamma(z)}\prod_p\Bigl(1+\frac{z-1}p\Bigr)\Bigl(1-\frac1p\Bigr)^{z-1}.$$
(The local theorem with $o(1)$ error gives this ratio; at $\lambda\approx2.8$ the error is not small. Is the corrected value at least closer than the Poisson one?)

**P11.4 (★★) CS application: finding an irreducible polynomial.** A finite field $\mathbb F_{2^n}$ is $\mathbb F_2[X]/(f)$ for an irreducible $f$ of degree $n$, and the standard way to find one is to try random monic polynomials until one is irreducible.

(a) Represent polynomials over $\mathbb F_2$ as Python integers (bit $i$ = coefficient of $X^i$) and implement multiplication modulo $f$ and gcd. Implement Ben-Or's irreducibility test: $f$ of degree $n$ is irreducible if and only if $\gcd(X^{2^i}-X,f)=1$ for $i=1,\dots,\lfloor n/2\rfloor$.

(b) The number of monic irreducibles of degree $n$ is $I_n=\frac1n\sum_{d\mid n}\mu(d)2^{n/d}$ (Gauss). Compute $2^nI_n^{-1}$, the expected number of tries, for $n=8,16,32,64,128$, and measure it with $300$ searches for $n=64$.

(c) The *number of irreducible factors* $\Omega(f)$ of a random monic $f$ of degree $n$ has the generating function $\sum_f z^{\Omega(f)}u^{\deg f}=\prod_d(1-zu^d)^{-I_d}$. Compute its exact law for $n=64$ and compare with the number of cycles of a random permutation of size $64$. Test (a)'s code on $1000$ random polynomials of degree $16$ by counting their irreducible factors by trial division, against the exact law.

## Solutions

### P11.1
**Idea.** The exact law comes from multiplying $n$ two-term polynomials. The Poisson law is right at the centre, $x=1$, and the residue $\psi(x)$ says how wrong it is elsewhere: below $1$ for many cycles, above $1$ for few.

In [ ]:
def cycles_exact(n, K=80):
    P = np.zeros(K); P[0] = 1.0
    for j in range(1, n+1):
        q = 1.0/j
        P[1:] = P[1:]*(1-q) + P[:-1]*q; P[0] *= 1-q
    return P

def count_cycles(p):
    seen = np.zeros(len(p), bool); c = 0
    for i in range(len(p)):
        if not seen[i]:
            c += 1; j = i
            while not seen[j]:
                seen[j] = True; j = p[j]
    return c

n, M = 1000, 20000
Kn = np.array([count_cycles(rng.permutation(n)) for _ in range(M)])
P = cycles_exact(n); Hn = sum(1/j for j in range(1, n+1))
print(f"H_n = {Hn:.4f}, empirical mean {Kn.mean():.4f}")
print(" k   empirical   exact    Poisson   corrected  exact/Poisson  psi(x)")
for k in range(1, 18):
    x = k/Hn; pois = poisson(Hn, k); psi = math.exp(-EG*(x-1))/Gamma(x)
    print(f"{k:2d}  {np.mean(Kn==k):.5f}  {P[k]:.5f}  {pois:.5f}  {pois*psi:.5f}  {P[k]/pois:8.3f}  {psi:8.3f}")

**Write-up.** Read the printed table. In the middle ($k\approx H_n\approx7.5$) the four columns agree. In the tails the Poisson law is wrong by a factor that the residue $\psi(x)=e^{-\gamma(x-1)}/\Gamma(x)$ captures: the Poisson law overestimates both tails (for $k=2$ and for $k=15$ the exact probability is about half the Poisson value), because $\psi(x)\le1$ with equality only at $x=1$: the cycle count is more concentrated than a Poisson variable with the same parameter. The corrected column follows the exact one, with a remaining error of a few per cent, of order $1/\log n$. The empirical frequencies agree with the exact law up to sampling noise, which checks Feller's coupling once more.

### P11.2
**Idea.** $|s(n,k)|=n!\,P(K_n=k)$, so this is the local theorem with $\lambda=\log n$ and residue $1/\Gamma$. The error term in the local theorem is $O(1/\log n)$, which is slow.

In [ ]:
def stirling1_row(n, K=40):
    row = [1] + [0]*K          # coefficients of z(z+1)...(z+m-1), starting from m=0
    for m in range(n):          # multiply by (z+m)
        new = [0]*(K+1)
        for k in range(K+1):
            new[k] = row[k]*m + (row[k-1] if k else 0)
        row = new
    return row

n = 2000
row = stirling1_row(n)
lf = math.lgamma(n)    # log (n-1)!
L, Hn = math.log(n), sum(1/j for j in range(1, n+1))
print(" k   ratio (log n)   ratio (H_n in Poisson part)")
for k in range(2, 21):
    logs = math.log(row[k])
    pred1 = lf + (k-1)*math.log(L) - math.lgamma(k) - math.log(Gamma(1+k/L))
    # corollary form: n! e^{-H_n} H_n^k/k! * e^{-gamma(x-1)}/Gamma(x), x=k/H_n
    x = k/Hn
    pred2 = math.lgamma(n+1) - Hn + k*math.log(Hn) - math.lgamma(k+1) - EG*(x-1) - math.log(Gamma(x))
    print(f"{k:2d}  {math.exp(logs-pred1):.4f}   {math.exp(logs-pred2):.4f}")

**Write-up.** With $\log n$ the ratio stays within about $11\%$ of $1$ for $2\le k\le20$, consistent with the $O(1/\log n)$ error of the local theorem ($1/\log2000\approx0.13$). The form with $H_n$ is better for $k$ around $13$ to $16$ but worse at both ends. The two forms are equivalent as $n\to\infty$ (they differ by factors such as $(H_n/\log n)^k$, which tend to $1$ uniformly for $k\le b\log n$), and convergence is slow for both.

### P11.3
**Idea.** A sieve adds $1$ to $\omega(m)$ for each prime $p\mid m$. The residue $\Psi_{\mathrm{unif}}$ is an Euler product that converges like $\prod(1+O(p^{-2}))$, so primes up to $10^6$ give it to several digits.

In [ ]:
N = 10**7
isp = np.ones(N+1, bool); isp[:2] = False
for p in range(2, int(N**0.5)+1):
    if isp[p]: isp[p*p::p] = False
primes = np.nonzero(isp)[0]
om = np.zeros(N+1, np.uint8)
for p in primes: om[p::p] += 1
cnt = np.bincount(om[1:])
lam = math.log(math.log(N))
pf = primes[primes < 10**6].astype(float)
def Psi_unif(x):
    w = x - 1
    return math.exp(np.sum(np.log1p(w/pf) + w*np.log1p(-1/pf))) / Gamma(x)
print(f"lambda = {lam:.4f}, mean of omega = {np.sum(np.arange(len(cnt))*cnt)/N:.4f}")
print(" k   proportion   Poisson   corrected   prop/Poisson  Psi_unif(k/lam)")
for k in range(1, len(cnt)):
    x = k/lam; po = poisson(lam, k); ps = Psi_unif(x)
    print(f"{k:2d}  {cnt[k]/N:.6f}  {po:.6f}  {po*ps:.6f}  {cnt[k]/N/po:8.3f}  {ps:8.3f}")

**Write-up.** The Poisson law badly overestimates both tails: $\omega(n)=1$ and $\omega(n)\ge6$ are much rarer than $\pi_\lambda$ says. The residue $\Psi_{\mathrm{unif}}$ goes in the right direction in both tails, and there the corrected column is much closer to the truth than the Poisson column (for $k=1$ it is almost exact). In the middle it does not help, and for the largest $k$ the truth is far more extreme than either: at $\lambda\approx2.8$ a local theorem with an $o(1)$ error is far from its limit, and $\omega(n)\le8$ for all $n\le10^7$ simply because $2\cdot3\cdots19\cdot23>10^7$.

### P11.4
**Idea.** (a) Polynomials over $\mathbb F_2$ are bit strings, addition is XOR, and reduction mod $f$ is "shift and XOR". In Ben-Or's test, $X^{2^i}\bmod f$ is obtained by squaring $i$ times. (b) Gauss's formula gives $I_n\approx2^n/n$, so about $n$ tries. (c) A random polynomial factors like a random permutation splits into cycles: the irreducible factors of degree $d$ play the role of the cycles of length $d$, and there are $I_d\approx2^d/d$ of them, just as there are $(d-1)!$ cycles of length $d$ on $d$ given points, a proportion $1/d$.

In [ ]:
def deg(a): return a.bit_length() - 1
def pmod(a, f):
    df = deg(f)
    while a and deg(a) >= df:
        a ^= f << (deg(a) - df)
    return a
def pmulmod(a, b, f):
    r = 0
    while b:
        if b & 1: r ^= a
        b >>= 1; a <<= 1
        if a >> deg(f) & 1: a ^= f
    return pmod(r, f)
def pgcd(a, b):
    while b: a, b = b, pmod(a, b)
    return a
def ben_or(f):
    n = deg(f); x = 2; y = x
    for i in range(1, n//2 + 1):
        y = pmulmod(y, y, f)              # X^(2^i) mod f
        if pgcd(f, y ^ x) != 1: return False
    return True

def mobius(n):
    m, p, res = n, 2, 1
    while p*p <= m:
        if m % p == 0:
            m //= p
            if m % p == 0: return 0
            res = -res
        p += 1
    return -res if m > 1 else res
def I(n): return sum(mobius(d) * 2**(n//d) for d in range(1, n+1) if n % d == 0) // n

# sanity: brute force count of irreducibles of degree 8
print("I_8 =", I(8), " Ben-Or count:", sum(ben_or((1 << 8) | c) for c in range(256)))
for n in (8, 16, 32, 64, 128):
    print(f"n={n:3d}: expected tries 2^n/I_n = {2**n / I(n):.3f}")
n = 64; tries = []
for _ in range(300):
    t = 0
    while True:
        t += 1
        f = (1 << n) | random.getrandbits(n)
        if ben_or(f): break
    tries.append(t)
print(f"n=64: mean tries over 300 searches = {np.mean(tries):.2f} (+- {np.std(tries)/math.sqrt(300):.2f})")

In [ ]:
from fractions import Fraction
def omega_poly_law(n, q=2):
    # coefficient of u^n in prod_d (1 - z u^d)^(-I_d), as a polynomial in z; then divide by q^n
    Id = {d: (sum(mobius(e) * q**(d//e) for e in range(1, d+1) if d % e == 0) // d) for d in range(1, n+1)}
    A = [[0]*(n+1) for _ in range(n+1)]   # A[m][k]: monic of degree m with k factors, using degrees processed so far
    A[0][0] = 1
    for d in range(1, n+1):
        B = [r[:] for r in A]
        for m in range(n+1):
            for k in range(n+1):
                if A[m][k] == 0: continue
                j, c = 1, Id[d]          # c = C(I_d + j - 1, j)
                while m + j*d <= n:
                    B[m+j*d][k+j] += A[m][k] * c
                    c = c * (Id[d] + j) // (j + 1); j += 1
        A = B
    return [Fraction(a, q**n) for a in A[n]]
law64 = omega_poly_law(64)
perm64 = cycles_exact(64)
print(" k  P(Omega=k), F_2[X] deg 64   P(K_64=k)")
for k in range(1, 15):
    print(f"{k:2d}   {float(law64[k]):.5f}            {perm64[k]:.5f}")
print("total probability:", float(sum(law64)))

def count_factors(f):
    # number of irreducible factors with multiplicity, by trial division by all irreducibles of degree <= deg/2
    k, d = 0, 1
    while deg(f) >= 2*d:
        for g in range(1 << d, 1 << (d+1)):
            if ben_or(g) or d == 1:
                while pmod(f, g) == 0 and deg(f) >= d:
                    f = pdiv(f, g); k += 1
        d += 1
    return k + (1 if deg(f) >= 1 else 0)
def pdiv(a, b):
    qt = 0
    while a and deg(a) >= deg(b):
        s = deg(a) - deg(b); qt ^= 1 << s; a ^= b << s
    return qt
law16 = omega_poly_law(16)
sample = [count_factors((1 << 16) | random.getrandbits(16)) for _ in range(1000)]
print(" k  sample freq (1000 polys of degree 16)   exact")
for k in range(1, 10):
    print(f"{k:2d}   {sample.count(k)/1000:.3f}    {float(law16[k]):.3f}")

**Write-up.** (a) The Ben-Or count of irreducibles of degree $8$ agrees with Gauss's formula, $I_8=30$. (b) The expected number of tries is very close to $n$, because $I_n=\frac{2^n}n+O(\frac{2^{n/2}}n)$, and the measured mean for $n=64$ agrees with $64$ within two standard errors. A random search for an irreducible polynomial of degree $n$ therefore costs about $n$ irreducibility tests, the same "one in $n$" as the probability that a random permutation of size $n$ is a single cycle. (c) The probability of exactly one factor (irreducibility) is $\frac1{64}$ in both models, but otherwise a random polynomial over $\mathbb F_2$ has noticeably *more* factors than a random permutation has cycles, and a much heavier right tail. The reason is repeated factors of small degree, which have no analogue for cycles: $X^2$ divides a random polynomial with probability $\frac14$, $X^{10}$ with probability $2^{-10}$. As $q\to\infty$ repeated factors become rare and the two laws coincide. For fixed $q$ the generating function above has a singularity of the same type as $\zeta(s)^z$, and the same contour analysis as in the Selberg–Delange theorem shows that $\Omega(f)$ converges mod-Poisson with parameters $\log n$, with an arithmetic residue that records the small factors, exactly as for integers. The sample of $1000$ factored polynomials agrees with the exact law, which also tests the factoring code.